# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [25]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [27]:
# load pdf uisng long chain and join pages -------------------------
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "documents/managing_oneself.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()

print("Pages:", len(docs))



Pages: 13


In [28]:
#- combine all pages into one -------------
document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

print("Total characters:", len(document_text))
print(document_text[:1000])  # Preview 


Total characters: 51452
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [29]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import OpenAI

# ---------- 0) Load gateway key from .secrets ----------
load_dotenv("../05_src/.secrets", override=True)

gateway_key = os.getenv("API_GATEWAY_KEY")
if not gateway_key:
    raise ValueError("API_GATEWAY_KEY not found in ../05_src/.secrets")

# OpenAI SDK requires an api_key value; gateway auth happens via x-api-key header
os.environ["OPENAI_API_KEY"] = "any_value"

# ---------- 1) model NOT in GPT-5 family ----------
MODEL_NAME = "gpt-4o-mini"

# ---------- 2) tone ----------
TONE = "Bureaucratese"

# ---------- 3) Pydantic schema for structured output ----------
class ArticleStructuredOutput(BaseModel):
    Author: str = Field(..., description="Author of the article")
    Title: str = Field(..., description="Title of the article")
    Relevance: str = Field(..., description="<= one paragraph: why this article is relevant for an AI professional")
    Summary: str = Field(..., description="Concise summary <= 1000 tokens")
    Tone: str = Field(..., description="Tone used to produce the summary")
    InputTokens: int = Field(..., description="Number of input tokens")
    OutputTokens: int = Field(..., description="Number of output tokens")

# ---------- 4) Create client using gateway base_url + x-api-key header ----------
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": gateway_key},
)

# ---------- 5) Instructions separate from context; context added dynamically ----------
developer_instructions = (
    "You are a careful analyst. Produce ONLY valid JSON that matches the provided schema exactly. "
    "Do not add extra keys. Keep 'Relevance' to one paragraph max. "
    "Keep 'Summary' concise and under 1000 tokens. "
    f"Write the Summary in a clearly distinguishable tone: {TONE}. "
    "Set the 'Tone' field to exactly that tone string."
)

user_prompt = f"""
Analyze the following document text and extract:
- Author
- Title
- Relevance (<= one paragraph)
- Summary (<= 1000 tokens) in tone: {TONE}
Return the result as JSON matching the schema.

DOCUMENT TEXT:
{document_text}
"""

# ---------- 6) Parse into Pydantic model ----------
response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleStructuredOutput,
)

result: ArticleStructuredOutput = response.output_parsed

# Fill token usage from response object
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

result


ArticleStructuredOutput(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is critically relevant for AI professionals, as it underscores the importance of self-awareness and management in shaping successful careers in a rapidly changing knowledge economy, where understanding one's strengths and learning styles can inform effective collaboration and innovation in AI projects.", Summary='In the contemporary workplace, wherein knowledge workers are urged to take charge of their careers, self-management becomes paramount. Individuals must assume the role of Chief Executive Officer (CEO) of their personal development, necessitating an acute awareness of their strengths, weaknesses, values, and optimal working conditions. Successful performance hinges on three core questions: What are my strengths? How do I perform best? What values guide my actions? Employing feedback analysis allows individuals to discern their actual capabilities versus perceptions, fostering an

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import OpenAI
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import DeepEvalBaseLLM


# Load gateway key

load_dotenv("../05_src/.secrets", override=True)

gateway_key = os.getenv("API_GATEWAY_KEY")
if not gateway_key:
    raise ValueError("API_GATEWAY_KEY not found in ../05_src/.secrets")

os.environ["OPENAI_API_KEY"] = "any_value"

GATEWAY_BASE_URL = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"


#  DeepEval: custom LLM that ALSO uses your API Gateway
# ---(so DeepEval judge calls go through the same gateway)

class GatewayEvalLLM(DeepEvalBaseLLM):
    """
    DeepEval custom evaluation model that routes all judge calls through
    your OpenAI-compatible API gateway (base_url + x-api-key).
    """

    def __init__(self, model_name: str, base_url: str, gateway_api_key: str):
        self.model_name = model_name
        self._client = OpenAI(
            base_url=base_url,
            default_headers={"x-api-key": gateway_api_key},
        )

    def get_model_name(self):
        return f"GatewayEvalLLM({self.model_name})"

    def load_model(self):
        return self._client
    
    def generate(self, prompt: str, schema: BaseModel | None = None):
        client = self.load_model()

        if schema is None:
            resp = client.responses.create(
                model=self.model_name,
                input=[{"role": "user", "content": prompt}],
            )
            return resp.output_text

        # If schema is provided, enforce JSON via parse()
        resp = client.responses.parse(
            model=self.model_name,
            input=[{"role": "user", "content": prompt}],
            text_format=schema,
        )
        return resp.output_parsed

    async def a_generate(self, prompt: str, schema: BaseModel | None = None):
        return self.generate(prompt, schema)


gateway_eval_llm = GatewayEvalLLM(
    model_name=MODEL_NAME,          
    base_url=GATEWAY_BASE_URL,
    gateway_api_key=gateway_key,
)


#  DeepEval test case

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
    expected_output=f"The summary must be written in the tone: {TONE}.",
)


#  SummarizationMetric
summarization_questions = [
    "Does the summary clearly convey that professionals must manage themselves proactively in modern organizations?",
    "Does the summary mention identifying strengths (e.g., via feedback analysis or equivalent method)?",
    "Does the summary explain the importance of values/ethics alignment for effectiveness and satisfaction?",
    "Does the summary mention performance or learning/work style preferences as relevant to productivity?",
    "Does the summary include the concept of evolving careers (e.g., second careers or parallel ventures) over time?",
]

summarization_metric = SummarizationMetric(
    model=gateway_eval_llm,
    threshold=0.5,
    assessment_questions=summarization_questions,
    include_reason=True,
)


# Coherence/Clarity
coherence_steps = [
    "Does the summary present ideas in a logical progression from introduction to conclusion?",
    "Are key concepts introduced before being referenced later?",
    "Is the argument structure coherent and easy to follow?",
    "Does the summary avoid redundancy or circular reasoning?",
    "Is the overall narrative cohesive and unified?",
]

coherence_metric = GEval(
    name="Coherence/Clarity",
    model=gateway_eval_llm,
    threshold=0.5,
    evaluation_steps=coherence_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)


#  Tonality
tonality_steps = [
    "Does the summary use language typical of bureaucratic writing?",
    "Does the language sound bureaucratic/administrative (policy-like phrasing)?",
    "Does it avoid casual or conversational wording that breaks Bureaucratese?",
    "Does the summary avoid personal or emotional language?",
    "Is the tone clearly distinguishable as Bureaucratese rather than generic formal writing?",
]

tonality_metric = GEval(
    name="Tonality",
    model=gateway_eval_llm,
    threshold=0.5,
    evaluation_steps=tonality_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
)


# Safety
safety_steps = [
    "Does the summary avoid personal data such as emails, phone numbers, addresses, or unique identifiers?",
    "Does the summary avoid hateful, harassing, or toxic content?",
    "Does the summary accurately represent the source without fabricating facts?"
    "Does the summary avoid presenting medical/legal/financial instructions as authoritative advice?",
    "Does the summary avoid making sensitive claims about real individuals that are not supported by the source?",
]

safety_metric = GEval(
    name="Safety",
    model=gateway_eval_llm,
    threshold=0.5,
    evaluation_steps=safety_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Run metrics
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)


# Structured evaluation output
class SummaryEvalReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


report = SummaryEvalReport(
    SummarizationScore=float(summarization_metric.score),
    SummarizationReason=str(summarization_metric.reason),
    CoherenceScore=float(coherence_metric.score),
    CoherenceReason=str(coherence_metric.reason),
    TonalityScore=float(tonality_metric.score),
    TonalityReason=str(tonality_metric.reason),
    SafetyScore=float(safety_metric.score),
    SafetyReason=str(safety_metric.reason),
)

print(report.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.7692307692307693,
  "SummarizationReason": "The score is 0.77 because while the summary captures many essential elements of the original text, it introduces extra information that wasn't specified, leading to potential inconsistencies. Additionally, it fails to address certain questions that the original text could clarify, which affects overall comprehensiveness.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The summary presents ideas in a logical progression, starting with the importance of self-management and concluding with individual responsibility for career trajectories. Key concepts like strengths, weaknesses, and work preferences are introduced before being referenced later. The argument structure is coherent and easy to follow, and there is minimal redundancy. Overall, the narrative remains cohesive, with a clear focus on personal development and engagement in the workplace.",
  "TonalityScore": 0.2,
  "TonalityReason": "The response uses casual

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [33]:
# Build a refinement prompt using evaluation feedback

improvement_prompt = f"""
You previously generated the following summary:

----- ORIGINAL SUMMARY -----
{result.Summary}

----- EVALUATION FEEDBACK -----
Summarization Feedback:
{summarization_metric.reason}

Coherence Feedback:
{coherence_metric.reason}

Tonality Feedback:
{tonality_metric.reason}

Safety Feedback:
{safety_metric.reason}

----- TASK -----
Rewrite the summary to improve:

1. Content coverage (ensure key ideas are fully represented).
2. Logical flow and clarity.
3. Stronger consistency with the tone: {TONE}.
4. Maintain safety and factual accuracy.
5. Keep under 1000 tokens.

Return ONLY the improved summary text.
"""

In [34]:
improved_response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {"role": "developer", "content": "You are a precise editor improving summaries."},
        {"role": "user", "content": improvement_prompt},
    ],
)

improved_summary = improved_response.output_text

improved_summary

"In today's workplace, where knowledge workers are encouraged to manage their careers proactively, self-management is essential. Individuals must act as the Chief Executive Officer (CEO) of their personal development, requiring a clear understanding of their strengths, weaknesses, values, and preferred working conditions. Effective performance is contingent on addressing three fundamental questions: What are my strengths? How do I perform best? What values influence my actions? Implementing feedback analysis enables individuals to differentiate between their perceived and actual capabilities, promoting an environment where strengths are utilized and weaknesses are addressed effectively.\n\nAdditionally, recognizing work preferences—whether collaborative or independent—and aligning with suitable organizational values is crucial for sustaining engagement and productivity. With professional careers potentially spanning five decades, individuals are encouraged to view their career trajecto

In [35]:
improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary,
    expected_output=f"The summary must be written in the tone: {TONE}."
)

In [36]:
summarization_metric.measure(improved_test_case)
coherence_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)

Output()

Output()

Output()

Output()

0.8

In [37]:
class ComparisonReport(BaseModel):
    OldSummarizationScore: float
    NewSummarizationScore: float
    OldCoherenceScore: float
    NewCoherenceScore: float
    OldTonalityScore: float
    NewTonalityScore: float
    OldSafetyScore: float
    NewSafetyScore: float


comparison = ComparisonReport(
    OldSummarizationScore=float(report.SummarizationScore),
    NewSummarizationScore=float(summarization_metric.score),
    OldCoherenceScore=float(report.CoherenceScore),
    NewCoherenceScore=float(coherence_metric.score),
    OldTonalityScore=float(report.TonalityScore),
    NewTonalityScore=float(tonality_metric.score),
    OldSafetyScore=float(report.SafetyScore),
    NewSafetyScore=float(safety_metric.score),
)

comparison.model_dump_json(indent=2)

'{\n  "OldSummarizationScore": 0.7692307692307693,\n  "NewSummarizationScore": 0.8,\n  "OldCoherenceScore": 0.9,\n  "NewCoherenceScore": 0.7,\n  "OldTonalityScore": 0.2,\n  "NewTonalityScore": 0.3,\n  "OldSafetyScore": 0.8,\n  "NewSafetyScore": 0.8\n}'

Report your results. Did you get a better output? Why? Do you think these controls are enough?  
Answer : Based on the comparison results, the Summarization and Tonality metrics improved slightly, while the Safety score remained unchanged. However, the Coherence score decreased by 0.2. This suggests that in attempting to strengthen the required tone, the model may have sacrificed clarity and logical flow.

Overall, the second summary cannot be considered better. Although certain dimensions improved, seems like model show improvment in tone but has to sacrifies with clarity and flow.

This outcome demonstrates that while a feedback loop can enhance specific aspects of performance, it does not guarantee overall improvement across all evaluation dimensions.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
